# 08 — Renderers, Alert Rules & Notification Channels

Covers Phase 6b components:

| Topic | What it covers |
|---|---|
| `HtmlRenderer.render()` | Single-run snapshot — metric table, status badges, bar chart |
| `HtmlRenderer.render_history()` | Multi-run time-series dashboard |
| `NoChartBackend` | Headless / offline rendering (tables only) |
| `ThresholdPolicy` | Stateless alert condition — custom threshold or delegates to `MetricResult.status` |
| `AlertRule` | Binds a metric name → policy → notification channels |
| `WebhookChannel` | HTTP POST via stdlib `urllib` — no extra dependencies |
| `EmailChannel` | SMTP via stdlib `smtplib` — optional HTML body via renderer |

> **Note:** `WebhookChannel` and `EmailChannel` calls are intercepted with `unittest.mock` so
> the notebook runs cleanly in CI without a live SMTP server or HTTP endpoint.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import email as email_lib
import json
from unittest.mock import MagicMock, patch

import numpy as np
import pandas as pd
from IPython.display import HTML, display

## Shared dataset and plan

Four weekly batches for a fraud detection model — the first three are healthy
(accuracy ≈ 0.88–0.92), the fourth is degraded (accuracy ≈ 0.65) to trigger alerts.

In [3]:
def make_batch(seed: int, accuracy_target: float = 0.90, n: int = 300) -> pd.DataFrame:
    """Synthetic binary classification batch with controllable accuracy."""
    rng_b = np.random.default_rng(seed)
    y_true = rng_b.integers(0, 2, n)
    # Flip a fraction of labels to hit the target accuracy
    y_pred = y_true.copy()
    flip = rng_b.random(n) < (1.0 - accuracy_target)
    y_pred[flip] = 1 - y_pred[flip]
    # Scores correlated with predictions (better AUC)
    y_score = np.where(
        y_pred == 1,
        rng_b.uniform(0.55, 1.0, n),
        rng_b.uniform(0.0, 0.45, n),
    )
    return pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "y_score": y_score})


reference = make_batch(seed=0, accuracy_target=0.92)

batches = [
    make_batch(seed=10, accuracy_target=0.92),  # healthy
    make_batch(seed=20, accuracy_target=0.90),  # healthy
    make_batch(seed=30, accuracy_target=0.88),  # healthy
    make_batch(seed=40, accuracy_target=0.65),  # degraded — triggers alert
]

for i, batch in enumerate(batches):
    acc = (batch["y_true"] == batch["y_pred"]).mean()
    label = "⚠️  degraded" if i == 3 else "✅ healthy"
    print(f"Batch {i + 1}: accuracy={acc:.3f}  {label}")

Batch 1: accuracy=0.917  ✅ healthy
Batch 2: accuracy=0.910  ✅ healthy
Batch 3: accuracy=0.867  ✅ healthy
Batch 4: accuracy=0.657  ⚠️  degraded


In [4]:
from ayn_ml.core.schema import TabularSchema
from ayn_ml.core.spec import MetricSpec, MonitoringPlan
from ayn_ml.runner import Runner

schema = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_score",
)

plan = MonitoringPlan(
    name="fraud_monitor",
    model_id="fraud_v3",
    model_version="3.1",
    data_schema=schema,
    metrics=[
        MetricSpec(name="accuracy", threshold=0.80, upper_bound=False),
        MetricSpec(name="f1"),
        MetricSpec(name="auc"),
    ],
    enable_profiling=False,
)

runner = Runner()

# Single report on the degraded batch — used throughout the notebook
report_degraded = runner.run(plan, batches[-1], reference=reference)

acc_result = next(r for r in report_degraded.results if r.spec.name == "accuracy")
print(f"accuracy={acc_result.value:.3f}  status={'PASS' if acc_result.status else 'FAIL'}")

accuracy=0.657  status=FAIL


---

## 1. HtmlRenderer — single report snapshot

`HtmlRenderer.render(report)` returns a complete HTML document as a string.
Pass it to `IPython.display.HTML` to view inline, write it to a `.html` file,
or attach it to an email body.

In [5]:
from ayn_ml.renderers import HtmlRenderer, NoChartBackend

renderer = HtmlRenderer()  # default: PlotlyBackend
html_snapshot = renderer.render(report_degraded)

print(f"HTML length : {len(html_snapshot):,} chars")
print(f"Metrics     : {len(report_degraded.results)}")
print(f"Errors      : {len(report_degraded.errors)}")

HTML length : 11,108 chars
Metrics     : 3
Errors      : 0


In [6]:
# Render inline — Plotly chart loads from CDN when connected to the internet
display(HTML(html_snapshot))

Metric,Type,Feature,Value,Status,Threshold,Effect Size
accuracy,,,0.6567,✗,0.8,—
f1,,,0.6567,—,—,—
auc,,,0.6673,—,—,—


### Headless mode — NoChartBackend

Use `NoChartBackend` when charts are not needed — CI pipelines, email bodies,
or environments without internet access.  Produces smaller, faster HTML.

In [7]:
headless_renderer = HtmlRenderer(charts=NoChartBackend())
html_headless = headless_renderer.render(report_degraded)

print(f"PlotlyBackend  (with charts) : {len(html_snapshot):>8,} chars")
print(f"NoChartBackend (tables only) : {len(html_headless):>8,} chars")

display(HTML(html_headless))

PlotlyBackend  (with charts) :   11,108 chars
NoChartBackend (tables only) :    3,089 chars


Metric,Type,Feature,Value,Status,Threshold,Effect Size
accuracy,,,0.6567,✗,0.8,—
f1,,,0.6567,—,—,—
auc,,,0.6673,—,—,—


---

## 2. render_history() — time-series dashboard

`render_history(reports)` accepts a list of `MonitoringReport` objects,
sorts them by `eval_timestamp`, and produces one time-series chart per metric
plus a latest-status summary table.

In [8]:
# Collect one MonitoringReport per batch
all_reports = [runner.run(plan, batch, reference=reference) for batch in batches]

print("Batch  accuracy  status")
print("-" * 30)
for i, r in enumerate(all_reports):
    acc = next((x.value for x in r.results if x.spec.name == "accuracy"), None)
    status = next((x.status for x in r.results if x.spec.name == "accuracy"), None)
    flag = "✅" if status else ("❌" if status is False else "—")
    print(f"  {i + 1}      {acc:.3f}     {flag}")

Batch  accuracy  status
------------------------------
  1      0.917     ✅
  2      0.910     ✅
  3      0.867     ✅
  4      0.657     ❌


In [9]:
html_history = HtmlRenderer().render_history(all_reports)
print(f"History dashboard: {len(html_history):,} chars")
display(HTML(html_history))

History dashboard: 27,112 chars


Metric,Feature,Latest Value,Status,Threshold
accuracy,,0.6567,✗,0.8
f1,,0.6567,—,—
auc,,0.6673,—,—


---

## 3. ThresholdPolicy — defining alert conditions

Two modes:

| Constructor | Behaviour |
|---|---|
| `ThresholdPolicy()` | Delegates to `MetricResult.status` — fires when the metric's own spec threshold is breached |
| `ThresholdPolicy(threshold=X, upper_bound=False)` | Custom lower bound — fires when `value < X` |
| `ThresholdPolicy(threshold=X, upper_bound=True)` | Custom upper bound — fires when `value > X` |

In [10]:
from ayn_ml.core.alert import ThresholdPolicy
from ayn_ml.core.result import MetricResult

# Construct two artificial MetricResult objects for demonstration
acc_pass = MetricResult(
    spec=MetricSpec(name="accuracy", threshold=0.80, upper_bound=False),
    value=0.92,
    status=True,  # 0.92 >= 0.80 → passes spec
)
acc_fail = MetricResult(
    spec=MetricSpec(name="accuracy", threshold=0.80, upper_bound=False),
    value=0.65,
    status=False,  # 0.65 < 0.80 → fails spec
)

# --- Policy 1: delegates to status ---
p_status = ThresholdPolicy()
print("ThresholdPolicy()  (delegates to status):")
print(f"  accuracy=0.92, status=True  → fires: {p_status.evaluate(acc_pass)}")  # False
print(f"  accuracy=0.65, status=False → fires: {p_status.evaluate(acc_fail)}")  # True

# --- Policy 2: custom lower bound (useful to alert at a different level than spec) ---
p_custom = ThresholdPolicy(threshold=0.70, upper_bound=False)
print("\nThresholdPolicy(threshold=0.70, upper_bound=False):")
print(f"  accuracy=0.92 → fires: {p_custom.evaluate(acc_pass)}")  # False (0.92 >= 0.70)
print(f"  accuracy=0.65 → fires: {p_custom.evaluate(acc_fail)}")  # True  (0.65 < 0.70)

# --- Policy 3: upper bound — useful for drift/error metrics ---
psi_result = MetricResult(
    spec=MetricSpec(name="psi", threshold=0.20, upper_bound=True),
    value=0.35,
    status=False,  # 0.35 > 0.20 → fails spec
)
p_drift = ThresholdPolicy(threshold=0.20, upper_bound=True)
print("\nThresholdPolicy(threshold=0.20, upper_bound=True)  (drift alert):")
print(f"  psi=0.35 → fires: {p_drift.evaluate(psi_result)}")  # True (0.35 > 0.20)

ThresholdPolicy()  (delegates to status):
  accuracy=0.92, status=True  → fires: False
  accuracy=0.65, status=False → fires: True

ThresholdPolicy(threshold=0.70, upper_bound=False):
  accuracy=0.92 → fires: False
  accuracy=0.65 → fires: True

ThresholdPolicy(threshold=0.20, upper_bound=True)  (drift alert):
  psi=0.35 → fires: True


---

## 4. AlertRule + Runner — end-to-end alert dispatch

`AlertRule` binds a metric name to a policy and a list of notification channels.
Pass `alert_rules=` to `Runner.run()` — fired alerts appear in `MonitoringReport.fired_alerts`.

In [11]:
from ayn_ml.core.alert import AlertRule

accuracy_rule = AlertRule(
    metric_name="accuracy",
    policy=ThresholdPolicy(threshold=0.80, upper_bound=False),  # fires when accuracy < 0.80
    channels=[],  # channels wired in later sections
)

# --- Degraded batch: accuracy ≈ 0.65 — alert fires ---
r_degraded = runner.run(plan, batches[-1], reference=reference, alert_rules=[accuracy_rule])
print(f"Degraded batch  →  fired alerts: {len(r_degraded.fired_alerts)}")
for fa in r_degraded.fired_alerts:
    print(f"  metric_name : {fa.metric_name}")
    print(f"  policy_type : {fa.policy_type}")
    print(f"  details     : {fa.details}")

# --- Healthy batch: accuracy ≈ 0.92 — no alert ---
r_healthy = runner.run(plan, batches[0], reference=reference, alert_rules=[accuracy_rule])
print(f"\nHealthy batch   →  fired alerts: {len(r_healthy.fired_alerts)}")

Degraded batch  →  fired alerts: 1
  metric_name : accuracy
  policy_type : threshold
  details     : {'value': 0.656667, 'threshold': 0.8, 'upper_bound': False}



Healthy batch   →  fired alerts: 0


### Multiple rules on the same metric

Two `AlertRule` objects can watch the same metric with different thresholds
(e.g. a warning level and a critical level) — each rule dispatches its own channels independently.

In [12]:
warning_ch = MagicMock()
critical_ch = MagicMock()

rules_tiered = [
    AlertRule(
        metric_name="accuracy",
        policy=ThresholdPolicy(threshold=0.80, upper_bound=False),  # warning: < 0.80
        channels=[warning_ch],
    ),
    AlertRule(
        metric_name="accuracy",
        policy=ThresholdPolicy(threshold=0.70, upper_bound=False),  # critical: < 0.70
        channels=[critical_ch],
    ),
]

r = runner.run(plan, batches[-1], reference=reference, alert_rules=rules_tiered)

acc_val = next(x.value for x in r.results if x.spec.name == "accuracy")
print(f"accuracy={acc_val:.3f}")
print(f"Fired alerts       : {len(r.fired_alerts)}")
print(f"warning_ch.write() : {warning_ch.write.called}")
print(f"critical_ch.write(): {critical_ch.write.called}")

accuracy=0.657
Fired alerts       : 2
warning_ch.write() : True
critical_ch.write(): True


---

## 5. WebhookChannel — HTTP POST notifications

`WebhookChannel` sends `MonitoringReport.to_dict()` as a JSON POST request
using only Python's stdlib `urllib`.  No third-party HTTP library required.

In [13]:
from ayn_ml.sinks.webhook import WebhookChannel

webhook = WebhookChannel(
    url="https://hooks.example.com/monitoring",
    extra_headers={"Authorization": "Bearer my-token", "X-Source": "ayn-ml"},
    timeout=10,
)

# Intercept the HTTP call without a live server
captured: dict = {}


def mock_urlopen(req, timeout=None):
    captured["url"] = req.full_url
    captured["method"] = req.get_method()
    captured["payload"] = json.loads(req.data.decode())
    captured["content_type"] = req.get_header("Content-type")
    captured["auth"] = req.get_header("Authorization")
    resp = MagicMock()
    resp.status = 200
    resp.__enter__ = lambda s: s
    resp.__exit__ = MagicMock(return_value=False)
    return resp


with patch("urllib.request.urlopen", side_effect=mock_urlopen):
    webhook.write(r_degraded)

print(f"URL          : {captured['url']}")
print(f"Method       : {captured['method']}")
print(f"Content-Type : {captured['content_type']}")
print(f"Auth header  : {captured['auth']}")
print(f"Payload keys : {list(captured['payload'].keys())}")

URL          : https://hooks.example.com/monitoring
Method       : POST
Content-Type : application/json
Auth header  : Bearer my-token
Payload keys : ['plan', 'context', 'results', 'errors', 'fired_alerts']


In [14]:
# Wire webhook into an AlertRule — write() is called only when the alert fires
webhook_channel = MagicMock()

rule_with_webhook = AlertRule(
    metric_name="accuracy",
    policy=ThresholdPolicy(threshold=0.80, upper_bound=False),
    channels=[webhook_channel],
)

# Degraded batch — alert fires → write() called
r = runner.run(plan, batches[-1], reference=reference, alert_rules=[rule_with_webhook])
print(f"Degraded batch — webhook.write() called: {webhook_channel.write.called}")

# Reset mock and run healthy batch — alert does not fire → write() NOT called
webhook_channel.reset_mock()
r = runner.run(plan, batches[0], reference=reference, alert_rules=[rule_with_webhook])
print(f"Healthy batch  — webhook.write() called: {webhook_channel.write.called}")

Degraded batch — webhook.write() called: True
Healthy batch  — webhook.write() called: False


---

## 6. EmailChannel — SMTP notifications

`EmailChannel` sends report summaries via SMTP using stdlib `smtplib`.
By default it sends a plain-text body.  Pass a renderer to get a multipart
email with both HTML and plain-text parts.

In [15]:
from ayn_ml.sinks.email import EmailChannel

email_channel = EmailChannel(
    host="smtp.example.com",
    to_addrs=["ops@example.com"],
    from_addr="monitoring@example.com",
    port=587,
    username="monitoring@example.com",
    password="app-password",
    use_tls=True,
)

# Mock SMTP — capture what would be sent
smtp_mock = MagicMock()
smtp_mock.__enter__ = lambda s: s
smtp_mock.__exit__ = MagicMock(return_value=False)
sent: dict = {}


def capture_sendmail(from_addr, to_addrs, message):
    sent["from"] = from_addr
    sent["to"] = to_addrs
    sent["message"] = message


smtp_mock.sendmail.side_effect = capture_sendmail

with patch("smtplib.SMTP", return_value=smtp_mock):
    email_channel.write(r_degraded)

# Decode the RFC 2047-encoded subject
import email.header

parsed = email_lib.message_from_string(sent["message"])
raw_subject = parsed.get("Subject", "")
decoded_parts = email.header.decode_header(raw_subject)
subject = "".join(part.decode(enc or "utf-8") if isinstance(part, bytes) else part for part, enc in decoded_parts)

print(f"From    : {sent['from']}")
print(f"To      : {sent['to']}")
print(f"Subject : {subject}")
print(f"STARTTLS: {smtp_mock.starttls.called}")
print(f"Login   : {smtp_mock.login.called}")
print(f"Content-Type: {parsed.get_content_type()}")

From    : monitoring@example.com
To      : ['ops@example.com']
Subject : [ayn-ml] fraud_monitor — 1 alert(s) fired
STARTTLS: True
Login   : True
Content-Type: text/plain


### HTML email — composing channels with renderers

Pass `renderer=HtmlRenderer(charts=NoChartBackend())` to get a multipart email
with an HTML body (Plotly charts disabled — CDN is unavailable in most SMTP contexts)
and an automatic plain-text fallback.

In [16]:
email_html_channel = EmailChannel(
    host="smtp.example.com",
    to_addrs=["ops@example.com"],
    renderer=HtmlRenderer(charts=NoChartBackend()),
)

smtp_mock2 = MagicMock()
smtp_mock2.__enter__ = lambda s: s
smtp_mock2.__exit__ = MagicMock(return_value=False)
sent2: dict = {}
smtp_mock2.sendmail.side_effect = lambda f, t, m: sent2.__setitem__("message", m)

with patch("smtplib.SMTP", return_value=smtp_mock2):
    email_html_channel.write(r_degraded)

msg2 = email_lib.message_from_string(sent2["message"])
parts = [part.get_content_type() for part in msg2.walk()]
print(f"Top-level Content-Type : {msg2.get_content_type()}")
print(f"All parts              : {parts}")

Top-level Content-Type : multipart/alternative
All parts              : ['multipart/alternative', 'text/plain', 'text/html']


---

## 7. Production pattern — stores + alerts + renderer

Typical production loop:
1. Each batch is persisted in a store (audit trail).
2. Alert rules evaluate every run — channels notify on breach.
3. After N runs, retrieve history from the store and render a dashboard.

In [17]:
from ayn_ml.stores import InMemoryStore

store = InMemoryStore()
ops_channel = MagicMock()  # replace with EmailChannel / WebhookChannel in production

alert_rule = AlertRule(
    metric_name="accuracy",
    policy=ThresholdPolicy(threshold=0.80, upper_bound=False),
    channels=[ops_channel],
)

prod_reports = []
for i, batch in enumerate(batches):
    r = runner.run(
        plan,
        batch,
        reference=reference,
        store=store,
        alert_rules=[alert_rule],
    )
    prod_reports.append(r)

    acc = next(x.value for x in r.results if x.spec.name == "accuracy")
    fired = "🔴 ALERT" if r.fired_alerts else "✅"
    print(f"Batch {i + 1}: accuracy={acc:.3f}  {fired}")

print(f"\nops_channel.write() call count : {ops_channel.write.call_count} (1 alert × 1 channel)")

Batch 1: accuracy=0.917  ✅
Batch 2: accuracy=0.910  ✅
Batch 3: accuracy=0.867  ✅
Batch 4: accuracy=0.657  🔴 ALERT

ops_channel.write() call count : 1 (1 alert × 1 channel)


In [18]:
# Render a history dashboard from the store
# read_history() returns flat dicts — collect unique run_ids to retrieve full reports
rows = store.read_history("fraud_v3")
seen: set = set()
reports_from_store = []
for row in rows:
    rid = row["run_id"]
    if rid not in seen:
        seen.add(rid)
        r = store.get_report(rid)
        if r is not None:
            reports_from_store.append(r)

print(f"Reports retrieved from store : {len(reports_from_store)}")

html_dashboard = HtmlRenderer().render_history(reports_from_store)
display(HTML(html_dashboard))

Reports retrieved from store : 4


Metric,Feature,Latest Value,Status,Threshold
accuracy,,0.6567,✗,0.8
f1,,0.6567,—,—
auc,,0.6673,—,—


---

## Summary

| Need | Recommended approach |
|---|---|
| View a single run inline | `HtmlRenderer().render(report)` + `display(HTML(...))` |
| Save a snapshot to disk | `Path("report.html").write_text(renderer.render(report))` |
| Multi-run dashboard | `HtmlRenderer().render_history(reports)` |
| Headless / no internet | `HtmlRenderer(charts=NoChartBackend())` |
| Alert on spec breach | `ThresholdPolicy()` — delegates to `MetricResult.status` |
| Alert on custom threshold | `ThresholdPolicy(threshold=X, upper_bound=False/True)` |
| HTTP POST on alert | `WebhookChannel(url=..., extra_headers=...)` |
| Email on alert | `EmailChannel(host=..., to_addrs=[...])` |
| Rich HTML email | `EmailChannel(..., renderer=HtmlRenderer(charts=NoChartBackend()))` |
| Always notify (every run) | Pass channel via `sinks=` — dispatches regardless of alerts |
| Notify only on alert | Pass channel via `AlertRule.channels` — dispatches only when alert fires |